In [ ]:
import pandas as pd
import numpy as np
import math
import re
from collections import defaultdict


class NaiveBayesSpamClassifier:
    def __init__(self):
        self.spam_count = 0
        self.ham_count = 0
        self.spam_words = defaultdict(int)
        self.ham_words = defaultdict(int)
        self.vocab = set()

    def preprocess_email(self, email_text):
        """Преобразует текст email в множество слов в нижнем регистре"""
        if pd.isna(email_text):
            return set()

        # Удаляем знаки пунктуации и переводим в нижний регистр
        words = re.findall(r"\b[a-zA-Zа-яА-Я]+\b", str(email_text).lower())
        return set(words)  # Учитываем только уникальные слова

    def train(self, emails, labels):
        """Обучает классификатор на размеченных данных"""
        for email, label in zip(emails, labels):
            words = self.preprocess_email(email)

            if label == 1 or label == "spam":  # спам
                self.spam_count += 1
                for word in words:
                    self.spam_words[word] += 1
                    self.vocab.add(word)
            else:  # не спам
                self.ham_count += 1
                for word in words:
                    self.ham_words[word] += 1
                    self.vocab.add(word)

    def predict(self, email_text):
        """Классифицирует email как спам (1) или не спам (0)"""
        words = self.preprocess_email(email_text)

        # Если нет данных для обучения
        if self.spam_count + self.ham_count == 0:
            return 0

        # Логарифмы априорных вероятностей
        log_p_spam = math.log(self.spam_count / (self.spam_count + self.ham_count))
        log_p_ham = math.log(self.ham_count / (self.spam_count + self.ham_count))

        # Вычисляем вероятности с сглаживанием Лапласа
        total_spam_words = sum(self.spam_words.values())
        total_ham_words = sum(self.ham_words.values())
        vocab_size = len(self.vocab)

        for word in words:
            # P(word|spam) с сглаживанием Лапласа
            p_word_spam = (self.spam_words.get(word, 0) + 1) / (
                total_spam_words + vocab_size + 1
            )
            log_p_spam += math.log(p_word_spam)

            # P(word|ham) с сглаживанием Лапласа
            p_word_ham = (self.ham_words.get(word, 0) + 1) / (
                total_ham_words + vocab_size + 1
            )
            log_p_ham += math.log(p_word_ham)

        # Решающее правило
        return 1 if log_p_spam > log_p_ham else 0

    def predict_proba(self, email_text):
        """Возвращает вероятность того, что email является спамом"""
        words = self.preprocess_email(email_text)

        if self.spam_count + self.ham_count == 0:
            return 0.5

        log_p_spam = math.log(self.spam_count / (self.spam_count + self.ham_count))
        log_p_ham = math.log(self.ham_count / (self.spam_count + self.ham_count))

        total_spam_words = sum(self.spam_words.values())
        total_ham_words = sum(self.ham_words.values())
        vocab_size = len(self.vocab)

        for word in words:
            p_word_spam = (self.spam_words.get(word, 0) + 1) / (
                total_spam_words + vocab_size + 1
            )
            p_word_ham = (self.ham_words.get(word, 0) + 1) / (
                total_ham_words + vocab_size + 1
            )

            log_p_spam += math.log(p_word_spam)
            log_p_ham += math.log(p_word_ham)

        # Преобразуем обратно в вероятности
        p_spam = math.exp(log_p_spam)
        p_ham = math.exp(log_p_ham)

        # Нормализуем
        total = p_spam + p_ham
        if total > 0:
            return p_spam / total
        return 0.5

    def evaluate(self, test_emails, test_labels):
        """Оценивает качество классификатора"""
        tp = 0  # True Positive (спам правильно классифицирован как спам)
        tn = 0  # True Negative (не спам правильно классифицирован как не спам)
        fp = 0  # False Positive (не спам ошибочно классифицирован как спам)
        fn = 0  # False Negative (спам ошибочно классифицирован как не спам)

        predictions = []

        for email, true_label in zip(test_emails, test_labels):
            predicted_label = self.predict(email)
            predictions.append(predicted_label)

            # Преобразуем метки к числовому формату для сравнения
            true_label_num = 1 if true_label == 1 or true_label == "spam" else 0

            if true_label_num == 1 and predicted_label == 1:
                tp += 1
            elif true_label_num == 0 and predicted_label == 0:
                tn += 1
            elif true_label_num == 0 and predicted_label == 1:
                fp += 1
            elif true_label_num == 1 and predicted_label == 0:
                fn += 1

        # Вычисляем метрики
        total = tp + tn + fp + fn
        accuracy = (tp + tn) / total if total > 0 else 0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0

        return {
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "accuracy": accuracy,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "predictions": predictions,
        }


def load_dataset(file_path):
    """Загружает датасет из CSV файла"""
    try:
        # Пробуем разные кодировки
        encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]

        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"Успешно загружено с кодировкой {encoding}: {len(df)} записей")
                print(f"Колонки: {df.columns.tolist()}")
                return df
            except UnicodeDecodeError:
                continue

        # Если ни одна кодировка не подошла
        print("Не удалось определить кодировку. Пробуем с ошибками...")
        df = pd.read_csv(file_path, encoding="utf-8", errors="ignore")
        return df

    except Exception as e:
        print(f"Ошибка загрузки файла: {e}")
        return None


def prepare_data(df):
    """Подготавливает данные для обучения"""
    # Проверяем наличие необходимых колонок
    required_columns = ["Message", "Spam/Ham"]
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        print(f"Отсутствуют колонки: {missing_columns}")
        print(f"Доступные колонки: {df.columns.tolist()}")
        return None, None

    # Обрабатываем метки
    def convert_label(label):
        if label in [1, "1", "spam", "Spam", "SPAM"]:
            return 1
        else:
            return 0

    emails = df["Message"].fillna("").tolist()
    labels = [convert_label(label) for label in df["Spam/Ham"]]

    # Статистика
    spam_count = sum(labels)
    ham_count = len(labels) - spam_count

    print(f"\nСтатистика датасета:")
    print(f"  Всего писем: {len(emails)}")
    print(f"  Спам: {spam_count} ({spam_count/len(emails)*100:.1f}%)")
    print(f"  Не спам: {ham_count} ({ham_count/len(emails)*100:.1f}%)")

    return emails, labels


def train_test_split(emails, labels, test_size=0.2, random_state=42):
    """Разделяет данные на обучающую и тестовую выборки"""
    np.random.seed(random_state)
    indices = np.random.permutation(len(emails))
    split_idx = int(len(emails) * (1 - test_size))

    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    train_emails = [emails[i] for i in train_indices]
    train_labels = [labels[i] for i in train_indices]
    test_emails = [emails[i] for i in test_indices]
    test_labels = [labels[i] for i in test_indices]

    return train_emails, train_labels, test_emails, test_labels


def main():
    print("Наивный байесовский классификатор спама")
    print("=" * 60)
    path = (
        "/home/julia/.cache/kagglehub/datasets/willyard/spam-email-dataset/versions/1"
    )
    file_path = os.path.join(path, "enron_spam_data.csv")
    df = load_dataset(file_path)

    if df is None:
        print("Не удалось загрузить датасет")
        return

    # Подготавливаем данные
    emails, labels = prepare_data(df)

    if emails is None:
        print("Не удалось подготовить данные")
        return

    # Разделяем на обучающую и тестовую выборки
    train_emails, train_labels, test_emails, test_labels = train_test_split(
        emails, labels, test_size=0.2
    )

    print(f"\nРазделение данных:")
    print(f"  Обучающая выборка: {len(train_emails)} писем")
    print(f"  Тестовая выборка: {len(test_emails)} писем")

    # Создаем и обучаем классификатор
    classifier = NaiveBayesSpamClassifier()
    classifier.train(train_emails, train_labels)

    print(f"\nСтатистика обучения:")
    print(f"  Спам-писем в обучении: {classifier.spam_count}")
    print(f"  Не спам-писем в обучении: {classifier.ham_count}")
    print(f"  Размер словаря: {len(classifier.vocab)}")

    # Тестируем
    results = classifier.evaluate(test_emails, test_labels)

    print(f"\nМАТРИЦА ОШИБОК:")
    print(f"  True Positive (TP):  {results['tp']} - спам правильно распознан")
    print(f"  True Negative (TN):  {results['tn']} - не спам правильно распознан")
    print(f"  False Positive (FP): {results['fp']} - не спам ошибочно помечен как спам")
    print(f"  False Negative (FN): {results['fn']} - спам ошибочно помечен как не спам")

    print(f"\nМЕТРИКИ КАЧЕСТВА:")
    print(f"  Точность (Accuracy):    {results['accuracy']:.3f}")
    print(f"  Чувствительность (Recall): {results['sensitivity']:.3f}")
    print(f"  Специфичность:          {results['specificity']:.3f}")
    print(f"  Точность (Precision):   {results['precision']:.3f}")


if __name__ == "__main__":
    main()